# Clinical cohort reproducibility analysis

This notebook reproduces the analysis workflow used for the independent clinical cohort.

The clinical dataset is not included in this repository due to privacy and ethical restrictions. To run this notebook, place a local file with the required structure at:

`data/clinical_data.csv`

The notebook is provided for transparency of the analytical workflow and should be adapted to local data before execution.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, spearmanr, chi2_contingency
from statsmodels.stats.multitest import multipletests
sns.set(style="whitegrid")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CLINICAL_DATA_PATH = DATA_DIR / "clinical_data.csv"

if not CLINICAL_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing clinical dataset: {CLINICAL_DATA_PATH}. "
        "The clinical dataset is not included in this repository due to privacy restrictions. "
        "Place a local file with the required structure at data/clinical_data.csv."
    )

df = pd.read_csv(CLINICAL_DATA_PATH)

In [ ]:
df['Group'] = df['Group'].str.strip()
df = df[df['Group'].isin(['МС', 'УЗД'])]

In [ ]:
columns_needed = [
    'Sample', 'Group', 'пол',
    'ИНСУЛИН', 'HOMA IR', 'ИМТ', 'ЛПНП', 'ЛПВП', 'ХОЛ',
    'АЛАТ', 'ЦРБв', 'АСАТ', 'ЩелФос', 'ГГТ', 'МОЧКИС',
    'КРЕАТИ', 'ТГ', 'Окружность талии', 'Возраст', 'ГЛ', 'БИЛ-ОБ', 'Гипотензивная терапия'
]
df = df[columns_needed]
# Переименование
rename_map = {
    'ИНСУЛИН': 'Insulin', 'HOMA IR': 'HOMA_IR', 'ИМТ': 'BMI',
    'ЛПНП': 'LDL', 'ХОЛ': 'Total_Cholesterol', 'АЛАТ': 'ALT',
    'АСАТ': 'AST', 'ГГТ': 'GGT', 'ЦРБв': 'CRP', 'МОЧКИС': 'Uric_Acid',
    'КРЕАТИ': 'Creatinine', 'ЩелФос': 'ALP', 'ЛПВП': 'HDL',
    'Возраст': 'Age', 'Окружность талии': 'Waist',
    'ГЛ': 'Glucose', 'ТГ': 'Triglycerides', 'БИЛ-ОБ': 'Bilirubin', 'Гипотензивная терапия': 'BP_therapy',
    'пол': 'Sex', 'Group': 'GROUP'
}
df = df.rename(columns=rename_map)

In [ ]:
df['BP_therapy'] = df['BP_therapy'].map({'Да': 1, 'Нет': 0})
df['Sex'] = df['Sex'].map({'м': 1, 'ж': 0})

In [ ]:
df['MetS'] = (df['GROUP'] == 'МС').astype(int)
print(f"N = {len(df)}")
print(df['GROUP'].value_counts())

In [ ]:
# Импорты (если еще не импортировали)
import pandas as pd
import numpy as np

# Загрузка данных (замените путь на актуальный)
df = pd.read_csv("/home/anastasia/Documents/1.Disser/Definitely_disser/anamnesis.csv")

# --- Ваши шаги по подготовке данных (как в original коде) ---
df['Group'] = df['Group'].str.strip()
df = df[df['Group'].isin(['МС', 'УЗД'])]

# Переименование колонок (ВАЖНО: используем те же названия, что и у вас)
rename_map = {
    'ИНСУЛИН': 'Insulin', 'HOMA IR': 'HOMA_IR', 'ИМТ': 'BMI',
    'ЛПНП': 'LDL', 'ХОЛ': 'Total_Cholesterol', 'АЛАТ': 'ALT',
    'АСАТ': 'AST', 'ГГТ': 'GGT', 'ЦРБв': 'CRP', 'МОЧКИС': 'Uric_Acid',
    'КРЕАТИ': 'Creatinine', 'ЩелФос': 'ALP', 'ЛПВП': 'HDL',
    'Возраст': 'Age', 'Окружность талии': 'Waist',
    'ГЛ': 'Glucose', 'ТГ': 'Triglycerides', 'БИЛ-ОБ': 'Bilirubin',
    'Гипотензивная терапия': 'BP_therapy', 'пол': 'Sex', 'Group': 'GROUP'
}
df = df.rename(columns=rename_map)

# Преобразуем бинарные переменные
df['BP_therapy'] = df['BP_therapy'].map({'Да': 1, 'Нет': 0})
df['Sex'] = df['Sex'].map({'м': 1, 'ж': 0})

# Создаем колонку MetS (1 - МС, 0 - УЗД)
df['MetS'] = (df['GROUP'] == 'МС').astype(int)

# --- Теперь правильные имена колонок для статистики (английские) ---
cols_for_stats = [
    'Age', 'BMI', 'Waist', 'BP_therapy', 'Sex',
    'Glucose', 'Insulin', 'HOMA_IR',
    'Triglycerides', 'HDL', 'LDL', 'Total_Cholesterol',
    'ALT', 'AST', 'GGT', 'CRP', 'Uric_Acid', 'Creatinine', 'Bilirubin'
]

# Функция для форматирования чисел
def format_number(x):
    if pd.isna(x):
        return 'NaN'
    if isinstance(x, float):
        if x.is_integer():
            return f'{int(x):d}'
        else:
            if abs(x) >= 10:
                return f'{x:.2f}'
            else:
                return f'{x:.3f}'
    return str(x)

# Собираем описательную статистику
desc_stats = []
for col in cols_for_stats:
    if col in df.columns:
        data = df[col].dropna()
        if len(data) > 0:
            stats = {
                'Variable': col,
                'N': len(data),
                'Mean': data.mean(),
                'SD': data.std(),
                'Min': data.min(),
                '1%': data.quantile(0.01),
                '5%': data.quantile(0.05),
                '50%': data.median(),
                '95%': data.quantile(0.95),
                '99%': data.quantile(0.99),
                'Max': data.max()
            }
            desc_stats.append(stats)

# Создаем DataFrame и форматируем
desc_df = pd.DataFrame(desc_stats)
numeric_cols = ['Mean', 'SD', 'Min', '1%', '5%', '50%', '95%', '99%', 'Max']
for col in numeric_cols:
    if col in desc_df.columns:
        desc_df[col] = desc_df[col].apply(format_number)

print("\n=== Descriptive Statistics for Whole Cohort (N=150) ===")
print(desc_df.to_string(index=False))

# Дополнительно: характеристика по группам
print("\n=== Group Summary ===")
print(df['GROUP'].value_counts().to_string())

# Статистика по группам для основных переменных
print("\n=== Basic Characteristics by Group ===")
for var in ['Age', 'BMI', 'Waist']:
    if var in df.columns:
        print(f"\n{var}:")
        for group in ['МС', 'УЗД']:
            data = df[df['GROUP'] == group][var].dropna()
            if len(data) > 0:
                print(f"  {group} (n={len(data)}): mean={data.mean():.2f}, sd={data.std():.2f}, "
                      f"median={data.median():.2f}, min={data.min():.2f}, max={data.max():.2f}")

In [ ]:
features_for_table = [
    "BMI", "Waist", "Glucose", "Insulin", "HOMA_IR",
    "Triglycerides", "HDL", "LDL", "Total_Cholesterol",
    "ALT", "AST", "GGT", "Bilirubin", "Creatinine", "Uric_Acid", "CRP"
]

def make_table1_formatted(df, target="MetS"):
    table = []
    p_values = []
    
    for var in features_for_table:
        g0 = df[df[target]==0][var].dropna()
        g1 = df[df[target]==1][var].dropna()
        
        stat, p = mannwhitneyu(g0, g1, alternative="two-sided")
        p_values.append(p)
        
        def fmt(x):
            return f"{x.median():.2f} [{x.quantile(0.25):.2f}–{x.quantile(0.75):.2f}]"
        
        table.append({
            "Variable": var,
            f"Control (n={len(g0)})": fmt(g0),
            f"MetS (n={len(g1)})": fmt(g1),
            "p-value": p
        })
    
    # FDR correction
    _, p_adj, _, _ = multipletests(p_values, method='fdr_bh')
    
    for i, row in enumerate(table):
        row["p_adj"] = p_adj[i]
    
    df_table = pd.DataFrame(table)
    
    # Форматируем p-value для вывода
    def format_pvalue(p):
        if p < 0.0001:
            return f"{p:.2e}"
        elif p < 0.001:
            return f"{p:.4f}"
        else:
            return f"{p:.3f}"
    
    df_table['p-value'] = df_table['p-value'].apply(format_pvalue)
    df_table['p_adj'] = df_table['p_adj'].apply(format_pvalue)
    
    return df_table

print("\n=== Table 1 (Formatted p-values) ===")
table1_formatted = make_table1_formatted(df)
print(table1_formatted.to_string(index=False))


In [ ]:
# ФОРМАТИРОВАННАЯ ТАБЛИЦА 
features = [
    "BMI", "Waist", "Glucose", "Insulin", "HOMA_IR",
    "Triglycerides", "HDL", "LDL", "Total_Cholesterol",
    "ALT", "AST", "GGT", "Bilirubin", "Creatinine", "Uric_Acid", "CRP"
]

def format_p(p):
    """Форматирует p-value в обычное число (не научный формат)"""
    if p < 0.0001:
        # Используем 15 десятичных знаков для очень маленьких чисел
        return f"{p:.15f}".rstrip('0').rstrip('.')
    elif p < 0.001:
        return f"{p:.6f}"
    elif p < 0.01:
        return f"{p:.5f}"
    elif p < 0.05:
        return f"{p:.4f}"
    else:
        return f"{p:.3f}"

results = []
p_vals = []

for var in features:
    ctrl = df[df['MetS']==0][var].dropna()
    mets = df[df['MetS']==1][var].dropna()
    stat, p = mannwhitneyu(ctrl, mets, alternative='two-sided')
    p_vals.append(p)
    results.append({
        'Variable': var,
        f'Control (n={len(ctrl)})': f"{ctrl.median():.2f} [{ctrl.quantile(0.25):.2f}–{ctrl.quantile(0.75):.2f}]",
        f'MetS (n={len(mets)})': f"{mets.median():.2f} [{mets.quantile(0.25):.2f}–{mets.quantile(0.75):.2f}]",
        'p-value': p
    })

# FDR correction
_, p_adj, _, _ = multipletests(p_vals, method='fdr_bh')
for i, r in enumerate(results):
    r['p_adj'] = p_adj[i]

df_results = pd.DataFrame(results)
df_results['p-value'] = df_results['p-value'].apply(format_p)
df_results['p_adj'] = df_results['p_adj'].apply(format_p)

print("\n=== СРАВНЕНИЕ ГРУПП (ОБЫЧНЫЙ ФОРМАТ) ===")
print(df_results.to_string(index=False))

In [ ]:
import pingouin as pg
import warnings
warnings.filterwarnings("ignore")

# Partial Spearman: контроль на Sex, Age, BMI
corr_features_partial = [
    "Waist", "Glucose", "Insulin", "HOMA_IR",
    "Triglycerides", "HDL", "LDL", "Total_Cholesterol",
    "ALT", "AST", "GGT", "Bilirubin", "Creatinine", "Uric_Acid", "CRP"
]

confounders = ["Sex", "Age"]

# 1. Обычный Spearman (baseline)
spearman_base = df[corr_features_partial].corr(method="spearman")
plt.figure(figsize=(12, 10))
sns.heatmap(spearman_base, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Spearman Correlation (unadjusted)", fontsize=14)
plt.tight_layout()
plt.show()

# 2. Partial Spearman (контроль Sex, Age, BMI)
partial_mat = pd.DataFrame(index=corr_features_partial, columns=corr_features_partial, dtype=float)

for i, c1 in enumerate(corr_features_partial):
    for j, c2 in enumerate(corr_features_partial):
        if i >= j:
            continue
        covar = [cv for cv in confounders if cv not in [c1, c2]]
        data = df[[c1, c2] + covar].dropna()
        if len(data) < 10:
            partial_mat.loc[c1, c2] = partial_mat.loc[c2, c1] = float("nan")
            continue
        res = pg.partial_corr(data=data, x=c1, y=c2, covar=covar, method="spearman")
        rho = res["r"].iloc[0]
        partial_mat.loc[c1, c2] = partial_mat.loc[c2, c1] = rho

plt.figure(figsize=(12, 10))
sns.heatmap(partial_mat.astype(float), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Partial Spearman (adjusted for Sex, Age)", fontsize=14)
plt.tight_layout()
plt.show()

print()
print("=== Partial Spearman Matrix ===")
print(partial_mat.astype(float).round(3))

# 3. Таблица пар (только значимые)
print()
print("=== Significant Partial Spearman (p < 0.05) ===")
results = []
for i, c1 in enumerate(corr_features_partial):
    for j, c2 in enumerate(corr_features_partial):
        if i >= j:
            continue
        covar = [cv for cv in confounders if cv not in [c1, c2]]
        data = df[[c1, c2] + covar].dropna()
        if len(data) < 10:
            continue
        res = pg.partial_corr(data=data, x=c1, y=c2, covar=covar, method="spearman")
        rho = res["r"].iloc[0]
        p = res["p_val"].iloc[0]
        results.append({"Var1": c1, "Var2": c2, "rho": round(rho, 3), "p": round(p, 4), "N": len(data), "sig": p < 0.05})

res_df = pd.DataFrame(results).sort_values("p")
print(res_df[res_df["sig"]].to_string(index=False))
print(f"Total pairs tested: {len(res_df)}, significant: {(res_df['p'] < 0.05).sum()}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, confusion_matrix

In [ ]:

def find_optimal_threshold(y_true, y_prob, metric="f1"):
    best_thresh, best_score = 0.5, 0
    for thresh in np.arange(0.1, 0.9, 0.02):
        y_pred = (y_prob >= thresh).astype(int)
        if metric == "f1":
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == "youden":
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            score = tpr - fpr
        if score > best_score:
            best_score = score
            best_thresh = thresh
    return best_thresh, best_score

def run_models_enhanced(df, features, title="Model"):
    df_model = df[features + ["MetS"]].dropna()
    print(f"N = {df_model.shape[0]}, Features: {len(features)}")

    X = df_model[features]
    y = df_model["MetS"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    results = []
    plt.figure(figsize=(7, 6))

    # Logistic Regression
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    logreg = LogisticRegression(max_iter=5000, class_weight="balanced")
    logreg.fit(X_train_scaled, y_train)
    y_prob = logreg.predict_proba(X_test_scaled)[:,1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"LogReg (AUC={auc:.3f})")

    opt_thresh, _ = find_optimal_threshold(y_test.values, y_prob, metric="f1")
    y_pred = (y_prob >= opt_thresh).astype(int)
    results.append({
        "Model": "LogReg",
        "AUC": auc,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0)
    })

    # Random Forest
    rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
    rf.fit(X_train, y_train)
    y_prob = rf.predict_proba(X_test)[:,1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"RF (AUC={auc:.3f})")

    opt_thresh, _ = find_optimal_threshold(y_test.values, y_prob, metric="f1")
    y_pred = (y_prob >= opt_thresh).astype(int)
    results.append({
        "Model": "RF",
        "AUC": auc,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0)
    })

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42,
        scale_pos_weight=len(y_train[y_train==0])/max(1, len(y_train[y_train==1]))
    )
    xgb.fit(X_train, y_train)
    y_prob = xgb.predict_proba(X_test)[:,1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"XGB (AUC={auc:.3f})")

    opt_thresh, _ = find_optimal_threshold(y_test.values, y_prob, metric="f1")
    y_pred = (y_prob >= opt_thresh).astype(int)
    results.append({
        "Model": "XGB",
        "AUC": auc,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0)
    })

    plt.plot([0, 1], [0, 1], 'k--', label="Random")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves - {title}")
    plt.legend()
    plt.show()

    for name, model in [("LogReg", logreg), ("RF", rf), ("XGB", xgb)]:
        if name == "LogReg":
            y_pred = model.predict(X_test_scaled)
        else:
            y_pred = model.predict(X_test)

        plt.figure()
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title(f"{title} - {name} Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.show()

    return pd.DataFrame(results)



In [ ]:
features_A = ["BMI", "Glucose", "Triglycerides", "HDL", "Waist", "BP_therapy"]
features_B = [
    "Age", "BMI", "Waist",
    "Glucose", "Insulin", "HOMA_IR",
    "Triglycerides", "HDL", "LDL", "Total_Cholesterol",
    "ALT", "AST", "GGT", "CRP",
    "Uric_Acid", "Creatinine", "BP_therapy"
]
features_C = [
    "Age",
    "ALT", "AST", "GGT", "CRP",
    "Uric_Acid", "Creatinine"
]
# Дедупликация (на всякий случай)
features_B = list(dict.fromkeys(features_B))
features_C = list(dict.fromkeys(features_C))
# Запуск моделей
res_A = run_models_enhanced(df, features_A)
res_B = run_models_enhanced(df, features_B)
res_C = run_models_enhanced(df, features_C)
print("\n=== ML Results (Small Dataset) ===")
print("A (Diagnostic):", res_A)
print("B (All):", res_B)
print("C (Non-diagnostic):", res_C)

In [ ]:
print("=== Scenario A (Diagnostic) ===")
res_A = run_models_enhanced(df, features_A, "A (Diagnostic)")
print(res_A.to_string(index=False))

In [ ]:
print("\n=== Scenario B (All) ===")
res_B = run_models_enhanced(df, features_B, "B (All)")
print(res_B.to_string(index=False))

In [ ]:
print("\n=== Scenario C (Non-diagnostic) ===")
res_C = run_models_enhanced(df, features_C, "C (Non-diagnostic)")
print(res_C.to_string(index=False))

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import numpy as np

def find_optimal_threshold(y_true, y_prob, metric="f1"):
    best_thresh, best_score = 0.5, 0
    for thresh in np.arange(0.1, 0.9, 0.02):
        y_pred = (y_prob >= thresh).astype(int)
        if metric == "f1":
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == "youden":
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            score = tpr - fpr
        if score > best_score:
            best_score = score
            best_thresh = thresh
    return best_thresh, best_score

def evaluate_models_cv_small(df, features, title="Model", n_splits=5):
    X = df[features].copy()
    y = df["MetS"].copy()
    
    X = X.select_dtypes(include=[np.number]).fillna(X.median())
    
    models = {
        "LogReg": Pipeline([("scaler", StandardScaler()), ("logit", LogisticRegression(max_iter=5000, class_weight="balanced"))]),
        "RF": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
        "XGB": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42, scale_pos_weight=len(y[y==0])/max(1,len(y[y==1])))
    }
    
    results = []
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    plt.figure(figsize=(7, 6))
    for name, model in models.items():
        y_prob = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
        auc = roc_auc_score(y, y_prob)
        fpr, tpr, _ = roc_curve(y, y_prob)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
        
        opt_thresh, opt_f1 = find_optimal_threshold(y.values, y_prob, metric="f1")
        y_pred = (y_prob >= opt_thresh).astype(int)
        prec = precision_score(y, y_pred, zero_division=0)
        rec = recall_score(y, y_pred, zero_division=0)
        f1 = f1_score(y, y_pred, zero_division=0)
        results.append({
            "Scenario": title,
            "Model": name,
            "AUC": f"{auc:.3f}",
            "Precision": f"{prec:.3f}",
            "Recall": f"{rec:.3f}",
            "F1": f"{f1:.3f}"
        })
    
    plt.plot([0, 1], [0, 1], "k--", label="Random")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves (CV) - {title}")
    plt.legend()
    plt.show()
    
    return pd.DataFrame(results)


In [ ]:
print("=" * 60)
print("CROSS-VALIDATION (k=5) - Small Dataset")
print("=" * 60)
cv_A = evaluate_models_cv_small(df, features_A, title="A (Diagnostic)")
cv_B = evaluate_models_cv_small(df, features_B, title="B (All)")
cv_C = evaluate_models_cv_small(df, features_C, title="C (Non-diagnostic)")

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

def run_cross_validation_metrics(df, features, title="Model"):
    """Выполняет 5-кратную кросс-валидацию с метриками и стандартным отклонением AUC"""
    df_model = df[features + ["MetS"]].dropna()
    X = df_model[features].values
    y = df_model["MetS"].values
    
    # Стандартизация для логистической регрессии
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Определяем модели
    models = {
        "LogReg": LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42),
        "RF": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
        "XGB": XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", 
            random_state=42, use_label_encoder=False,
            scale_pos_weight=len(y[y==0])/max(1, len(y[y==1]))
        )
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    results = []
    
    for name, model in models.items():
        # Выбираем данные (масштабированные для LogReg)
        if name == "LogReg":
            X_use = X_scaled
        else:
            X_use = X
        
        # Собираем предсказания и AUC по каждой фолде
        auc_scores = []
        y_true_all = []
        y_prob_all = []
        
        for train_idx, val_idx in cv.split(X_use, y):
            X_train, X_val = X_use[train_idx], X_use[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            model_clone = clone(model)
            model_clone.fit(X_train, y_train)
            
            y_prob = model_clone.predict_proba(X_val)[:, 1]
            auc = roc_auc_score(y_val, y_prob)
            auc_scores.append(auc)
            
            y_true_all.extend(y_val)
            y_prob_all.extend(y_prob)
        
        # Среднее и стандартное отклонение AUC
        auc_mean = np.mean(auc_scores)
        auc_std = np.std(auc_scores)
        
        # Оптимальный порог для F1 на всех данных
        best_thresh, _ = find_optimal_threshold(np.array(y_true_all), np.array(y_prob_all), metric="f1")
        y_pred_opt = (np.array(y_prob_all) >= best_thresh).astype(int)
        
        results.append({
            "Model": name,
            "AUC_mean": f"{auc_mean:.4f}",
            "AUC_std": f"{auc_std:.4f}",
            "AUC_scores": [f"{s:.3f}" for s in auc_scores],
            "Accuracy": f"{accuracy_score(y_true_all, y_pred_opt):.4f}",
            "Precision": f"{precision_score(y_true_all, y_pred_opt, zero_division=0):.4f}",
            "Recall": f"{recall_score(y_true_all, y_pred_opt, zero_division=0):.4f}",
            "F1": f"{f1_score(y_true_all, y_pred_opt, zero_division=0):.4f}"
        })
    
    results_df = pd.DataFrame(results)
    print(f"\n=== 5-Fold Cross-Validation Results ({title}) ===")
    print(results_df[['Model', 'AUC_mean', 'AUC_std', 'Accuracy', 'Precision', 'Recall', 'F1']].to_string(index=False))
    
    # Дополнительный вывод с AUC по фолдам
    print(f"\n--- Detailed AUC per fold ---")
    for res in results:
        print(f"{res['Model']}: {res['AUC_scores']} (mean={res['AUC_mean']} ± {res['AUC_std']})")
    
    return results_df

In [ ]:
print("\n" + "="*60)
print("CROSS-VALIDATION WITH AUC ± STD")
print("="*60)

# Полная версия с метриками
cv_full_A = run_cross_validation_metrics(df, features_A, "A (Diagnostic)")
cv_full_B = run_cross_validation_metrics(df, features_B, "B (All)")
cv_full_C = run_cross_validation_metrics(df, features_C, "C (Non-diagnostic)")

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import make_scorer, roc_auc_score

def run_cross_validation(df, features, title="Model"):
    """Выполняет 5-кратную кросс-валидацию для трех моделей"""
    df_model = df[features + ["MetS"]].dropna()
    X = df_model[features]
    y = df_model["MetS"]
    
    # Стандартизация для логистической регрессии
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Определяем модели
    models = {
        "LogReg": LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42),
        "RF": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
        "XGB": XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", 
            random_state=42, use_label_encoder=False,
            scale_pos_weight=len(y[y==0])/max(1, len(y[y==1]))
        )
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scoring = make_scorer(roc_auc_score)
    
    results = []
    for name, model in models.items():
        if name == "LogReg":
            scores = cross_val_score(model, X_scaled, y, cv=cv, scoring=scoring)
        else:
            scores = cross_val_score(model, X, y, cv=cv, scoring=scoring)
        
        results.append({
            "Model": name,
            "CV_AUC_mean": f"{scores.mean():.4f}",
            "CV_AUC_std": f"{scores.std():.4f}",
            "CV_scores": [f"{s:.3f}" for s in scores]
        })
    
    results_df = pd.DataFrame(results)
    print(f"\n=== 5-Fold CV Results ({title}) ===")
    print(results_df.to_string(index=False))
    
    return results_df

# Пример вызова:
cv_A = run_cross_validation(df, features_A, "A (Diagnostic)")

In [ ]:
cv_B = run_cross_validation(df, features_B, "B (All)")

In [ ]:
cv_C = run_cross_validation(df, features_C, "C (Non-diagnostic)")

In [ ]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

def feature_importance_analysis(df, features, title="Model"):
    """Feature importance для RF и XGBoost"""
    X = df[features].copy()
    y = df["MetS"].copy()
    X = X.select_dtypes(include=[np.number]).fillna(X.median())
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # Random Forest
    rf = RandomForestClassifier(n_estimators=300, random_state=42)
    rf.fit(X_train, y_train)
    rf_imp = pd.DataFrame({
        "Feature": features,
        "RF_importance": rf.feature_importances_
    }).sort_values("RF_importance", ascending=False)
    
    # XGBoost
    xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42)
    xgb.fit(X_train, y_train)
    xgb_imp = pd.DataFrame({
        "Feature": features,
        "XGB_importance": xgb.feature_importances_
    }).sort_values("XGB_importance", ascending=False)
    
    # Объединяем
    combined = rf_imp.merge(xgb_imp, on="Feature")
    combined = combined.sort_values("RF_importance", ascending=False)
    
    # График
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    axes[0].barh(combined["Feature"], combined["RF_importance"], color="forestgreen")
    axes[0].set_xlabel("Importance")
    axes[0].set_title(f"{title} - Random Forest")
    axes[0].invert_yaxis()
    
    axes[1].barh(combined["Feature"], combined["XGB_importance"], color="orange")
    axes[1].set_xlabel("Importance")
    axes[1].set_title(f"{title} - XGBoost")
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    return combined

In [ ]:
print("=== Feature Importance - Scenario A ===")
fi_A = feature_importance_analysis(df, features_A, "A (Diagnostic)")

In [ ]:
print("=== Feature Importance - Scenario B ===")
fi_B = feature_importance_analysis(df, features_B, "B (All)")

In [ ]:
print("=== Feature Importance - Scenario C ===")
fi_C = feature_importance_analysis(df, features_C, "C (Non-diagnostic)")